In [1]:
# --- libraries & model -------------------------------------------------------
import collections
import random
import numpy as np
import torch
import evaluate as eval_lib
from datasets import load_dataset
from tqdm import tqdm
from transformers import (
    AlbertTokenizerFast,
    AlbertForQuestionAnswering,
)

MODEL_DIR = "../albert_squad2_finetuned/checkpoint-37500"

tokenizer = AlbertTokenizerFast.from_pretrained(MODEL_DIR)
model     = AlbertForQuestionAnswering.from_pretrained(MODEL_DIR)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# --- feature preparation -----------------------------------------------------
def prepare_features(examples, max_length: int = 384, doc_stride: int = 128):
    pad_on_right = tokenizer.padding_side == "right"
    tokenised = tokenizer(
        examples["question" if pad_on_right else "context"],
        examples["context"  if pad_on_right else "question"],
        truncation="longest_first",
        max_length=max_length,
        stride=doc_stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )
    mapping = tokenised.pop("overflow_to_sample_mapping")
    tokenised["example_id"] = [examples["id"][idx] for idx in mapping]
    return tokenised

# --- post-processing ----------------------------------------------------------
def postprocess(preds, feats, examples):
    start_logits, end_logits = preds
    per_ex = collections.defaultdict(list)
    for idx, ex_id in enumerate(feats["example_id"]):
        per_ex[ex_id].append(idx)

    final = collections.OrderedDict()
    n_best = 20
    max_len = 30

    for ex in examples:
        ex_id = ex["id"]
        prelim = []
        for fi in per_ex[ex_id]:
            offsets     = feats["offset_mapping"][fi]
            s_log, e_log = start_logits[fi], end_logits[fi]

            for s in np.argsort(s_log)[-n_best:]:
                for e in np.argsort(e_log)[-n_best:]:
                    if e < s or (e - s + 1) > max_len:
                        continue
                    if offsets[s] is None or offsets[e] is None:
                        continue
                    prelim.append({
                        "score": s_log[s] + e_log[e],
                        "start": offsets[s][0],
                        "end":   offsets[e][1],
                    })

        if prelim:
            best = max(prelim, key=lambda p: p["score"])
            final[ex_id] = {
                "text":  ex["context"][best["start"]: best["end"]],
                "score": best["score"],
            }
        else:
            final[ex_id] = {"text": "", "score": 0.0}
    return final

# --- Step 1: Load data, tokenize, predict -------------------------------------
print("\nLoading dataset…")
examples = load_dataset("squad")["validation"]

print(f"Tokenising {len(examples)} examples …")
features = prepare_features(examples)

print(f"Running model on {len(features['input_ids'])} features…")
batch_size = 8
all_start_logits, all_end_logits = [], []

for i in tqdm(range(0, len(features["input_ids"]), batch_size), desc="Predicting"):
    batch = {
        k: torch.tensor(v[i:i + batch_size]).to(device)
        for k, v in features.items() if k in ["input_ids", "attention_mask"]
    }
    with torch.no_grad():
        outputs = model(**batch)
    all_start_logits.extend(outputs.start_logits.cpu().numpy())
    all_end_logits.extend(outputs.end_logits.cpu().numpy())

print("\nFinished predictions! Ready for evaluation.")

/Users/seanhall/Desktop/NLPFinalProject/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



Loading dataset…
Tokenising 10570 examples …
Running model on 10808 features…


Predicting: 100%|██████████| 1351/1351 [29:52<00:00,  1.33s/it]


Finished predictions! Ready for evaluation.


In [2]:
# --- Post-process predictions and evaluate metrics ------------------------------
print("\nPost-processing predictions…")
predictions = postprocess((all_start_logits, all_end_logits), features, examples)

metric = eval_lib.load("squad")

references = [{"id": ex["id"], "answers": ex["answers"]} for ex in examples]
preds_list = [{"id": ex["id"], "prediction_text": predictions[ex["id"]]["text"]} for ex in examples]

results = metric.compute(predictions=preds_list, references=references)
em_key = "exact_match" if "exact_match" in results else "exact"

print("\nResults on SQuAD 1.1 validation set:")
print(f"Exact Match: {results[em_key]:.2f}")
print(f"F1 Score:    {results['f1']:.2f}")

# --- Show some wrong examples -------------------------------------------------
wrong = [
    {"question": ex["question"],
     "pred": predictions[ex["id"]]["text"],
     "gold": ex["answers"]["text"][0]}
    for ex in examples
    if predictions[ex["id"]]["text"] not in ex["answers"]["text"]
]

if wrong:
    print("\n3 Examples of Wrong Predictions:")
    for i, s in enumerate(random.sample(wrong, min(3, len(wrong))), 1):
        print(f"\nExample {i}:")
        print(f"Question : {s['question']}")
        print(f"Predicted: '{s['pred']}'")
        print(f"Expected : '{s['gold']}'")
else:
    print("\nNo wrong predictions found!")



Post-processing predictions…

Results on SQuAD 1.1 validation set:
Exact Match: 76.39
F1 Score:    83.22

3 Examples of Wrong Predictions:

Example 1:
Question : Where did the first shipment of minerals ship from?
Predicted: 'Kilifi'
Expected : 'Base Titanium, a subsidiary of Base resources of Australia'

Example 2:
Question : What replica was used for player introductions?
Predicted: 'Golden Gate Bridge'
Expected : 'the Golden Gate Bridge.'

Example 3:
Question : Which leaders did the Islamic extremists attack?
Predicted: 'Muslim states'
Expected : 'apostate'
